# 大样本均值置信区间的模拟

> 对应代码：`MathStatsCode/code_in_notes/Chap.9/CI_large_sample.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：对均值为0的双峰分布，用正态临界值构造置信区间，比较 n=10 与 n=100 时的覆盖率。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.9`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.9")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

本 notebook 考察**大样本置信区间**在非正态总体下的表现：
- 定义程序 `CIz`：生成双峰数据 `x = d·(N(μ,1)) + (1-d)·(N(-μ,1))`（μ=3，即由两个分隔的正态分量混合而成，真实均值恰为0），然后用**正态临界值** `invnormal(1-α/2)` 构造区间 x̄ ± z_{α/2}·s/√n；
- 返回区间是否覆盖真值0的指示变量；
- 先 `simulate ... : CIz, obs(10)` 重复10000次、每次样本量10——此时样本量太小、总体严重非正态，**覆盖率明显低于95%**（t 区间或许也未必准确，而 z 区间更差）；
- 再 `simulate ... : CIz, obs(100)` 重复10000次、每次样本量100——根据中心极限定理 x̄ 已近似正态，覆盖率回升到 95% 附近。
该实验说明：使用正态近似（大样本方法）需要样本量足够大，否则区间估计的名义置信水平无法保证。


In [ ]:
%%stata
clear
set seed 97
set more off
cap program drop CIz
program define CIz, rclass
	syntax [, obs(integer 100) mu(real 3) confident_level(real 0.95)]
	quietly{
		drop _all
		set obs `obs'
		tempvar d x 
		tempname lb ub alpha
		local `alpha'=1-`confident_level'
		gen `d'=runiform()<0.5
		gen `x'=`d'*(rnormal()+`mu')+(1-`d')*(rnormal()-`mu')
		su `x'
		local `lb'=r(mean)-invnormal(1-``alpha''/2)*r(sd)/sqrt(`obs')
		local `ub'=r(mean)+invnormal(1-``alpha''/2)*r(sd)/sqrt(`obs')
		return scalar lb=``lb''
		return scalar ub=``ub''
		return scalar include_true=(``lb''<=0 & ``ub''>=0)
	}
end

simulate it=r(include_true), reps(10000): CIz, obs(10)
su
simulate it=r(include_true), reps(10000): CIz, obs(100)
su
